# 00. 선형대수 기초 (Linear Algebra Foundations)

이번 단원에서는 딥러닝에 필요한 **선형대수 기초**를 배워보겠습니다.

## 학습 목표
- 벡터와 행렬의 기하학적 의미
- 내적(Dot Product)과 행렬곱의 직관
- 선형 변환과 차원 변환의 이해
- 텐서의 개념과 PyTorch 연결

## 왜 이 단원이 필요한가?

딥러닝의 모든 연산은 **행렬 연산**입니다. 신경망을 이해하려면 선형대수의 직관이 필수입니다.

- **텐서**: PyTorch의 모든 데이터는 텐서 (벡터와 행렬의 일반화)
- **Linear Layer**: 행렬곱으로 입력을 출력으로 변환
- **내적**: 가중치 벡터와 입력 벡터의 유사도 측정
- **선형 변환**: 신경망이 데이터를 학습 가능한 표현으로 변환

이 단원을 이해하면 텐서 연산과 신경망 구조가 명확해집니다.

## 이 단원을 배우기 전에

**고등학교 수학 복습**: 벡터와 행렬의 기본 연산을 기억하시나요?

## 이 단원 다음에는

**다음 단원 (00a)**: 선형대수를 이해했으니, 이제 미적분 기초를 배워봅시다.

---
# 1. 직관적 이해 (Why)
---

## 1.1 왜 선형대수가 필요한가?

### 딥러닝 = 행렬 연산

신경망의 모든 계산은 행렬 곱셈입니다:

```python
# 간단한 신경망 예시
import torch
import torch.nn as nn

# Linear Layer: 입력 10차원 → 출력 3차원
layer = nn.Linear(10, 3)
x = torch.randn(32, 10)  # 배치 32개, 각 10차원

# Forward: 행렬 곱셈!
output = layer(x)  # (32, 10) @ (10, 3) = (32, 3)
```

**핵심**: `nn.Linear`는 행렬 곱셈 `W @ x + b`를 수행합니다!

### 기하학적 직관

- **벡터**: 데이터의 한 샘플 (숫자들의 배열)
- **행렬**: 변환 규칙 (공간을 바꾸는 함수)
- **행렬곱**: 데이터를 새로운 공간으로 투영
- **내적**: 두 벡터의 유사도 측정

**의미**: 신경망은 데이터를 "더 잘 분리되는 공간"으로 변환하는 행렬을 학습합니다!


---
# 2. 수학적 기초 (What)
---

## 2.1 벡터의 기하학적 의미

벡터는 **방향과 크기**를 가진 화살표입니다.

2차원 벡터: $\vec{v} = \begin{bmatrix} v_1 \\ v_2 \end{bmatrix}$

**기하학적 해석**:
- **크기**: $\|\vec{v}\| = \sqrt{v_1^2 + v_2^2}$
- **방향**: 원점에서 $(v_1, v_2)$로 가는 화살표

### 머신러닝에서의 의미

**입력 데이터**: 이미지, 텍스트, 사용자 특성 → 모두 벡터  
**가중치 벡터**: 각 뉴런이 학습하는 "패턴"


In [ ]:
# 벡터 시각화
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1) 기본 벡터
ax1 = axes[0]
v = np.array([3, 4])
ax1.arrow(0, 0, v[0], v[1], head_width=0.3, head_length=0.3,
          fc='blue', ec='blue', linewidth=2, label='벡터 v = [3, 4]')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-1, 5)
ax1.set_ylim(-1, 5)
ax1.set_aspect('equal')
ax1.legend()
ax1.set_title('벡터의 기하학적 표현')

# 2) 여러 벡터 비교
ax2 = axes[1]
vectors = [np.array([3, 4]), np.array([-2, 3]), np.array([1, -2])]
colors = ['blue', 'red', 'green']
for v, color in zip(vectors, colors):
    ax2.arrow(0, 0, v[0], v[1], head_width=0.3, head_length=0.3,
              fc=color, ec=color, linewidth=2, label=f'v = [{v[0]}, {v[1]}]')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-3, 4)
ax2.set_ylim(-3, 5)
ax2.set_aspect('equal')
ax2.legend()
ax2.set_title('여러 벡터 비교')

plt.tight_layout()
plt.show()

print("벡터의 성질:")
for v in vectors:
    print(f"- 벡터 {v}: 크기 = {np.linalg.norm(v):.2f}")
print("→ 머신러닝: 각 데이터 샘플이 하나의 벡터")


## 2.2 내적(Dot Product)의 의미

두 벡터의 내적:

$$\vec{a} \cdot \vec{b} = a_1 b_1 + a_2 b_2 = \|\vec{a}\| \|\vec{b}\| \cos(\theta)$$

**기하학적 의미**: 두 벡터의 **정렬도(alignment)** 측정
- 같은 방향 → 내적 최대
- 수직 → 내적 = 0
- 반대 방향 → 내적 최소

### 머신러닝 연결

Linear Layer: `output = input @ weight.T + bias`

**내적 = 유사도**: 가중치 벡터와 입력 벡터가 얼마나 비슷한 패턴인가?


In [ ]:
# 내적의 기하학적 의미 시각화
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

a = np.array([3, 0])
vectors_b = [
    (np.array([2, 0]), "같은 방향"),
    (np.array([0, 2]), "수직"),
    (np.array([-2, 0]), "반대 방향")
]

for i, (b, title) in enumerate(vectors_b):
    ax = axes[i]
    ax.arrow(0, 0, a[0], a[1], head_width=0.3, head_length=0.3,
             fc='blue', ec='blue', linewidth=2, label='벡터 a')
    ax.arrow(0, 0, b[0], b[1], head_width=0.3, head_length=0.3,
             fc='red', ec='red', linewidth=2, label='벡터 b')
    
    dot = np.dot(a, b)
    ax.set_title(f'{title}\n내적 = {dot:.1f}')
    ax.set_xlim(-3, 4)
    ax.set_ylim(-1, 3)
    ax.set_aspect('equal')
    ax.grid(True, alpha=0.3)
    ax.axhline(y=0, color='k', linestyle='--', linewidth=0.5)
    ax.axvline(x=0, color='k', linestyle='--', linewidth=0.5)
    ax.legend()

plt.tight_layout()
plt.show()

print("내적의 의미:")
print("- 같은 방향: 내적 큼 → 유사한 패턴")
print("- 수직: 내적 0 → 무관한 패턴")
print("- 반대 방향: 내적 음수 → 반대 패턴")
print("→ 가중치 벡터와 입력의 내적이 크면 해당 뉴런 활성화!")


## 2.3 행렬의 기하학적 의미

행렬은 **공간을 변환**하는 함수입니다.

행렬곱: $A\vec{x} = \begin{bmatrix} a_{11} & a_{12} \\ a_{21} & a_{22} \end{bmatrix} \begin{bmatrix} x_1 \\ x_2 \end{bmatrix}$

**열벡터 관점**: 행렬의 열벡터들의 선형결합

$$A\vec{x} = x_1 \begin{bmatrix} a_{11} \\ a_{21} \end{bmatrix} + x_2 \begin{bmatrix} a_{12} \\ a_{22} \end{bmatrix}$$

### 머신러닝 연결

Linear Layer는 행렬 곱셈:
```python
output = input @ weight.T + bias
```

각 출력 = 가중치 벡터와 입력의 내적  
**의미**: 입력을 새로운 공간으로 투영하는 변환


---
# 3. PyTorch 구현 (How)
---

## 3.1 텐서로 표현하기

PyTorch에서는 모든 데이터를 텐서(tensor)로 표현:

- **스칼라**: 0차원 텐서
- **벡터**: 1차원 텐서
- **행렬**: 2차원 텐서
- **텐서**: 3차원 이상

텐서는 벡터와 행렬을 일반화한 다차원 배열입니다!


In [ ]:
# 텐서로 벡터와 행렬 표현
import torch

print("=== 차원별 텐서 ===\n")

# 스칼라 (0차원)
scalar = torch.tensor(3.14)
print(f"스칼라: {scalar}, shape: {scalar.shape}, 차원: {scalar.dim()}D")

# 벡터 (1차원)
vector = torch.tensor([1, 2, 3])
print(f"벡터: {vector}, shape: {vector.shape}, 차원: {vector.dim()}D")

# 행렬 (2차원)
matrix = torch.tensor([[1, 2, 3], [4, 5, 6]])
print(f"행렬:\n{matrix}")
print(f"shape: {matrix.shape}, 차원: {matrix.dim()}D")

# 3차원 텐서
tensor_3d = torch.randn(2, 3, 4)
print(f"\n3D 텐서 shape: {tensor_3d.shape}, 차원: {tensor_3d.dim()}D")
print("→ 2개의 행렬, 각 3×4 크기\n")

print("머신러닝 의미:")
print("- 이미지: (배치, 채널, 높이, 너비)")
print("- 모든 데이터는 차원을 가진 텐서로 표현!")


## 3.2 Linear Layer 직접 구현

Linear Layer는 **행렬곱 + 편향**입니다:

$$y = Wx + b$$

여기서:
- $x$: 입력 벡터 (in_features)
- $W$: 가중치 행렬 (out_features × in_features)
- $b$: 편향 벡터 (out_features)
- $y$: 출력 벡터 (out_features)

**PyTorch의 nn.Linear**는 이 수식을 구현합니다!


In [ ]:
# Linear Layer 구현 비교
import torch
import torch.nn as nn

print("=== Linear Layer 구현 비교 ===\n")

in_features = 5
out_features = 3
batch_size = 4

x = torch.randn(batch_size, in_features)
print(f"입력 x shape: {x.shape}")

# 방법 1: PyTorch nn.Linear
linear_layer = nn.Linear(in_features, out_features)
y1 = linear_layer(x)
print(f"nn.Linear 출력 shape: {y1.shape}")

# 방법 2: 수동 구현 (행렬곱 + 편향)
W = linear_layer.weight.data
b = linear_layer.bias.data
y2 = x @ W.T + b
print(f"수동 구현 출력 shape: {y2.shape}")
print(f"결과 일치: {torch.allclose(y1, y2)} ✓\n")

print("핵심:")
print("- nn.Linear = 행렬곱 (W @ x) + 편향 (b)")
print("- 각 출력은 가중치 벡터와 입력의 내적!")
print("- 신경망의 가장 기본적인 연산!")


---
# 4. 핵심 요약
---

## 이번 단원에서 배운 내용

### 1. 벡터와 행렬
- **벡터**: 방향과 크기를 가진 화살표 (데이터 샘플)
- **행렬**: 공간을 변환하는 함수 (선형 변환)
- **행렬곱**: 열벡터들의 선형결합으로 공간 변환
- **내적**: 두 벡터의 정렬도 측정 (유사도)

### 2. 머신러닝 연결
- **텐서**: 벡터와 행렬을 일반화한 다차원 배열
- **Linear Layer**: 행렬곱 + 편향으로 입력을 출력으로 변환
- **내적의 의미**: 가중치 벡터와 입력 벡터의 유사도

### 3. PyTorch 구현
- `torch.tensor`: 스칼라, 벡터, 행렬, 텐서 생성
- `@` 연산자: 행렬곱
- `nn.Linear`: y = Wx + b 구현

## 다음 단원 미리보기

**다음 단원 (00a)**: 미적분 기초
- 선형대수로 구조를 이해했으니, 이제 **미적분**을 배웁니다
- **미분**: 함수의 변화율 (기울기)
- **Gradient**: 손실을 줄이는 방향
- **Gradient Descent**: 학습 알고리즘의 수학적 원리

**다음 단원 (01)**: 텐서 기초
- 이제 본격적으로 PyTorch의 **텐서**를 배웁니다
- 실제 데이터를 텐서로 변환하는 방법

수학 기초가 탄탄해졌으니, 이제 PyTorch를 배울 준비가 되었습니다!
